In [ ]:
# ============================================================
#  🔍 INFERÊNCIA — ResNet-50 | Dataset DATAP + trainLabels.csv
#  📂 Pastas: No_DR / Mild / Moderate / Severe / Proliferate_DR
#  📄 CSV   : trainLabels.csv (gabarito)
#  👉 Cole tudo em UMA célula do Colab e rode
# ============================================================

import subprocess
subprocess.run(['pip', 'install', '-q', 'albumentations', 'opencv-python',
                'scikit-learn', 'tqdm', 'seaborn'])

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ── Imports ───────────────────────────────────────────────────
import os, gc, warnings
import torch
import torch.nn as nn
import torchvision.models as models
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from torch.utils.data        import Dataset, DataLoader
from torch.cuda.amp          import autocast
from sklearn.metrics         import (classification_report,
                                     confusion_matrix,
                                     roc_auc_score,
                                     accuracy_score,
                                     cohen_kappa_score,
                                     precision_recall_fscore_support)
from sklearn.preprocessing   import label_binarize
from tqdm.auto               import tqdm
import albumentations        as A
from albumentations.pytorch  import ToTensorV2

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ── Device ────────────────────────────────────────────────────
device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = (device.type == 'cuda')
print(f'Dispositivo : {device}')
if device.type == 'cuda':
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ══════════════════════════════════════════════════════════════
#  ⚙️  CONFIGURAÇÕES — ajuste se necessário
# ══════════════════════════════════════════════════════════════

# Pasta raiz do DATAP
DATAP_ROOT   = '/content/drive/MyDrive/RETINOPATIA/DATAP'

# CSV com gabarito (coluna "image" ou "filename" + coluna "level" ou "label")
CSV_PATH     = os.path.join(DATAP_ROOT, 'trainLabels.csv')

# Modelo treinado (ResNet-50 gerado pelo treino anterior)
MODEL_PATH   = '/content/drive/MyDrive/RETINOPATIA/resnet50_retinopatia_final.pth'

# Pasta para salvar resultados
OUTPUT_DIR   = '/content/drive/MyDrive/RETINOPATIA/inferencia_datap'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Pastas a IGNORAR dentro do DATAP
IGNORE_DIRS  = {'TRABALHO PROF GUILHERME', 'validacao_resultados',
                'checkpoints_resnet50', 'resnet50_resultados'}

IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_WORKERS  = 2

N_CLASSES    = 5
CLASS_NAMES  = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

# Mapeamento pasta → label numérico
FOLDER_TO_LABEL = {
    'No_DR':0,         'no_dr':0,       '0':0,
    'Mild':1,          'mild':1,        '1':1,
    'Moderate':2,      'moderate':2,    '2':2,
    'Severe':3,        'severe':3,      '3':3,
    'Proliferate_DR':4,'proliferate_dr':4,'Proliferative':4,'4':4,
}

print(f'\nConfig:')
print(f'  DATAP   : {DATAP_ROOT}')
print(f'  CSV     : {CSV_PATH}')
print(f'  Modelo  : {MODEL_PATH}')
print(f'  Saída   : {OUTPUT_DIR}')

# ══════════════════════════════════════════════════════════════
#  PASSO 1 — Inspecionar o CSV e entender sua estrutura
# ══════════════════════════════════════════════════════════════

print('\n' + '='*55)
print('PASSO 1 — Lendo trainLabels.csv')
print('='*55)

csv_raw = pd.read_csv(CSV_PATH)
print(f'Shape   : {csv_raw.shape}')
print(f'Colunas : {list(csv_raw.columns)}')
print(f'\nPrimeiras linhas:')
print(csv_raw.head(5).to_string())

# Detecta automaticamente as colunas de imagem e label
def detect_columns(df):
    img_col   = None
    label_col = None
    for c in df.columns:
        cl = c.lower().strip()
        if cl in ('image','filename','file','name','img','id','imageid','image_name'):
            img_col = c
        if cl in ('level','label','class','grade','severity','diagnosis','target'):
            label_col = c
    # Fallback: primeira coluna = imagem, segunda = label
    if img_col   is None: img_col   = df.columns[0]
    if label_col is None: label_col = df.columns[1] if len(df.columns) > 1 else df.columns[0]
    return img_col, label_col

img_col, label_col = detect_columns(csv_raw)
print(f'\n✅ Coluna de imagem detectada : "{img_col}"')
print(f'✅ Coluna de label detectada  : "{label_col}"')
print(f'   Valores únicos de label    : {sorted(csv_raw[label_col].unique())}')

# ══════════════════════════════════════════════════════════════
#  PASSO 2 — Montar DataFrame de imagens com caminhos reais
# ══════════════════════════════════════════════════════════════

print('\n' + '='*55)
print('PASSO 2 — Mapeando imagens nas pastas DATAP')
print('='*55)

# Estratégia: varre as pastas válidas e indexa TODAS as imagens por nome de arquivo
# Depois cruza com o CSV para ter gabarito

# Indexar todas as imagens disponíveis: {nome_sem_ext: caminho_completo}
img_index = {}
pastas_validas = []

for d in sorted(os.listdir(DATAP_ROOT)):
    full_d = os.path.join(DATAP_ROOT, d)
    if not os.path.isdir(full_d):
        continue
    if d in IGNORE_DIRS:
        print(f'  ⏭️  Ignorando: {d}/')
        continue
    if d not in FOLDER_TO_LABEL:
        print(f'  ⚠️  Pasta não mapeada (ignorada): {d}/')
        continue

    pastas_validas.append(d)
    label = FOLDER_TO_LABEL[d]
    imgs  = [f for f in os.listdir(full_d)
             if f.lower().endswith(('.jpg','.jpeg','.png'))]
    for img_file in imgs:
        stem = os.path.splitext(img_file)[0]   # nome sem extensão
        img_index[stem]      = os.path.join(full_d, img_file)
        img_index[img_file]  = os.path.join(full_d, img_file)  # com extensão também

    print(f'  ✅ {d:<20} label={label}  {len(imgs):>7} imagens')

print(f'\nTotal indexado: {len(img_index)//2} imagens únicas')

# ── Cruzar CSV com imagens ─────────────────────────────────────
records = []
not_found = 0

for _, row in csv_raw.iterrows():
    img_name  = str(row[img_col]).strip()
    label_val = row[label_col]

    # Tentar encontrar a imagem: com ou sem extensão
    path = (img_index.get(img_name)
            or img_index.get(img_name + '.jpg')
            or img_index.get(img_name + '.jpeg')
            or img_index.get(img_name + '.png')
            or img_index.get(os.path.splitext(img_name)[0]))

    if path is None:
        not_found += 1
        continue

    # Normalizar label para inteiro 0-4
    try:
        label_int = int(float(label_val))
    except:
        label_int = FOLDER_TO_LABEL.get(str(label_val), -1)

    if label_int < 0 or label_int >= N_CLASSES:
        continue

    records.append({'image': path, 'label': label_int, 'name': img_name})

df = pd.DataFrame(records)
print(f'\nImagens cruzadas com CSV : {len(df)}')
print(f'Não encontradas no CSV   : {not_found}')

if len(df) == 0:
    # Fallback: usa diretamente as pastas, sem CSV
    print('\n⚠️  Nenhuma imagem cruzada com CSV.')
    print('   Modo fallback: usando pastas diretamente (sem gabarito CSV).')
    data = []
    for d in pastas_validas:
        label = FOLDER_TO_LABEL[d]
        folder_path = os.path.join(DATAP_ROOT, d)
        for img_file in os.listdir(folder_path):
            if img_file.lower().endswith(('.jpg','.jpeg','.png')):
                data.append({'image': os.path.join(folder_path, img_file),
                             'label': label, 'name': img_file})
    df = pd.DataFrame(data)
    print(f'   Imagens encontradas nas pastas: {len(df)}')

print('\nDistribuição final:')
for i, name in enumerate(CLASS_NAMES):
    n = (df['label'] == i).sum()
    print(f'  [{i}] {name:<18} {n:>7} imagens  ({100*n/len(df):.1f}%)')

# ══════════════════════════════════════════════════════════════
#  PASSO 3 — Dataset e DataLoader
# ══════════════════════════════════════════════════════════════

transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])


class RetinaInferenceDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df        = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = cv2.imread(row['image'])
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, int(row['label']), row['name']


dataset = RetinaInferenceDataset(df, transform)
loader  = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=False,
    persistent_workers=False,
    prefetch_factor=2 if NUM_WORKERS > 0 else None
)
print(f'\nDataLoader: {len(loader)} batches ({len(dataset)} imagens)')

# ══════════════════════════════════════════════════════════════
#  PASSO 4 — Carregar modelo ResNet-50
# ══════════════════════════════════════════════════════════════

print('\n' + '='*55)
print('PASSO 4 — Carregando modelo ResNet-50')
print('='*55)

assert os.path.exists(MODEL_PATH), \
    f'❌ Modelo não encontrado: {MODEL_PATH}\n' \
    f'   Rode o notebook de treino primeiro!'

def build_resnet50(n_classes=5):
    m           = models.resnet50(weights=None)
    in_features = m.fc.in_features
    m.fc        = nn.Sequential(
        nn.Dropout(p=0.4),
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.Dropout(p=0.3),
        nn.Linear(512, n_classes)
    )
    return m

model = build_resnet50(N_CLASSES)
sd    = torch.load(MODEL_PATH, map_location=device)
sd    = {k.replace('_orig_mod.', ''): v for k, v in sd.items()}
model.load_state_dict(sd, strict=True)
model = model.to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'✅ Modelo carregado — {n_params:.1f} M parâmetros')

# ══════════════════════════════════════════════════════════════
#  PASSO 5 — Inferência
# ══════════════════════════════════════════════════════════════

print('\n' + '='*55)
print('PASSO 5 — Inferência')
print('='*55)

all_labels, all_preds, all_probs, all_names = [], [], [], []

with torch.no_grad():
    for images, labels, names in tqdm(loader, desc='Inferindo'):
        images = images.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP):
            logits = model(images)
        probs = torch.softmax(logits.float(), dim=1).cpu().numpy()

        all_labels.extend(labels.numpy())
        all_preds.extend(probs.argmax(axis=1))
        all_probs.append(probs)
        all_names.extend(names)

        del images, logits, probs

gc.collect()
if device.type == 'cuda':
    torch.cuda.empty_cache()

all_labels = np.array(all_labels)
all_preds  = np.array(all_preds)
all_probs  = np.vstack(all_probs)

print(f'✅ Inferência concluída — {len(all_labels)} imagens')

# ══════════════════════════════════════════════════════════════
#  PASSO 6 — Métricas
# ══════════════════════════════════════════════════════════════

print('\n' + '='*55)
print('PASSO 6 — Métricas')
print('='*55)

acc   = accuracy_score(all_labels, all_preds)
kappa = cohen_kappa_score(all_labels, all_preds, weights='quadratic')

labels_bin = label_binarize(all_labels, classes=list(range(N_CLASSES)))
try:
    auc = roc_auc_score(labels_bin, all_probs, average='macro', multi_class='ovr')
except Exception as e:
    auc = float('nan')
    print(f'  ⚠️  AUC não calculado: {e}')

prec, rec, f1, sup = precision_recall_fscore_support(
    all_labels, all_preds, average=None,
    labels=list(range(N_CLASSES)), zero_division=0
)
f1_macro    = f1.mean()
f1_weighted = precision_recall_fscore_support(
    all_labels, all_preds, average='weighted', zero_division=0
)[2]

print(f'\n{"Métrica":<22} {"Valor":>10}')
print('-' * 35)
print(f'{"Acurácia":<22} {acc:>10.4f}  ({acc*100:.1f}%)')
print(f'{"F1-Macro":<22} {f1_macro:>10.4f}')
print(f'{"F1-Weighted":<22} {f1_weighted:>10.4f}')
print(f'{"AUC-ROC (macro)":<22} {auc:>10.4f}')
print(f'{"Kappa Quadrático":<22} {kappa:>10.4f}')
print('-' * 35)

print('\n📋 Classification Report:')
print(classification_report(all_labels, all_preds,
                             target_names=CLASS_NAMES, digits=4))

print('\n📊 F1 por classe:')
print(f'  {"Classe":<18} {"Prec":>7} {"Rec":>7} {"F1":>7} {"Suporte":>9}')
print('  ' + '-'*50)
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:<18} {prec[i]:>7.4f} {rec[i]:>7.4f} {f1[i]:>7.4f} {int(sup[i]):>9}')

# ══════════════════════════════════════════════════════════════
#  PASSO 7 — Matriz de Confusão
# ══════════════════════════════════════════════════════════════

cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(cm_norm, annot=False, ax=ax, cmap='Blues', vmin=0, vmax=1,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, linecolor='white')

for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        val   = cm_norm[i, j]
        color = 'white' if val > 0.5 else 'black'
        ax.text(j+0.5, i+0.38, f'{val:.2f}',
                ha='center', va='center', fontsize=10,
                fontweight='bold', color=color)
        ax.text(j+0.5, i+0.65, f'n={cm[i,j]}',
                ha='center', va='center', fontsize=7.5,
                color=color, alpha=0.8)

ax.set_title(
    f'Matriz de Confusão — ResNet-50 | Dataset DATAP\n'
    f'Acc={acc:.4f}  |  Kappa={kappa:.4f}  |  AUC={auc:.4f}',
    fontweight='bold', fontsize=12
)
ax.set_ylabel('Classe Real', fontsize=11)
ax.set_xlabel('Classe Predita', fontsize=11)
ax.tick_params(axis='x', rotation=30)
ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
path_cm = os.path.join(OUTPUT_DIR, 'inferencia_confusion_matrix.png')
plt.savefig(path_cm, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Salvo: {path_cm}')

# ══════════════════════════════════════════════════════════════
#  PASSO 8 — F1 por classe (gráfico)
# ══════════════════════════════════════════════════════════════

fig, ax = plt.subplots(figsize=(9, 4.5))
x      = np.arange(N_CLASSES)
bars   = ax.bar(x, f1, color='#2E75B6', alpha=0.85, edgecolor='white', linewidth=0.8)
ax.axhline(f1_macro, color='#DD8452', ls='--', lw=1.8, label=f'F1-macro={f1_macro:.3f}')

for bar, val in zip(bars, f1):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=11)
ax.set_ylabel('F1-Score', fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_title('F1-Score por Classe — ResNet-50 | DATAP', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
path_f1 = os.path.join(OUTPUT_DIR, 'inferencia_f1_por_classe.png')
plt.savefig(path_f1, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Salvo: {path_f1}')

# ══════════════════════════════════════════════════════════════
#  PASSO 9 — Distribuição de confiança
# ══════════════════════════════════════════════════════════════

max_prob = all_probs.max(axis=1)
correct  = all_labels == all_preds

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(max_prob[correct],  bins=40, alpha=0.72, color='#2ca02c',
        label=f'Corretas  (n={correct.sum()})', density=True)
ax.hist(max_prob[~correct], bins=40, alpha=0.72, color='#d62728',
        label=f'Erradas   (n={(~correct).sum()})', density=True)

m_cor = max_prob[correct].mean()
m_err = max_prob[~correct].mean() if (~correct).sum() > 0 else 0
ax.axvline(m_cor, color='#2ca02c', ls='--', lw=1.8, label=f'Média correta: {m_cor:.3f}')
ax.axvline(m_err, color='#d62728', ls='--', lw=1.8, label=f'Média errada : {m_err:.3f}')

ax.set_xlabel('Confiança (probabilidade máxima)', fontsize=11)
ax.set_ylabel('Densidade', fontsize=11)
ax.set_title('Distribuição de Confiança — Acertos vs Erros', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
path_conf = os.path.join(OUTPUT_DIR, 'inferencia_confianca.png')
plt.savefig(path_conf, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Salvo: {path_conf}')

# ══════════════════════════════════════════════════════════════
#  PASSO 10 — Salvar predições em CSV
# ══════════════════════════════════════════════════════════════

df_out = pd.DataFrame({
    'imagem':          all_names,
    'label_real':      all_labels,
    'classe_real':     [CLASS_NAMES[l] for l in all_labels],
    'pred':            all_preds,
    'classe_predita':  [CLASS_NAMES[p] for p in all_preds],
    'correto':         (all_labels == all_preds).astype(int),
    'confianca':       max_prob.round(4),
    **{f'prob_{CLASS_NAMES[i].replace(" ","_")}': all_probs[:, i].round(4)
       for i in range(N_CLASSES)}
})

path_csv = os.path.join(OUTPUT_DIR, 'inferencia_predicoes.csv')
df_out.to_csv(path_csv, index=False)
print(f'\n💾 Predições salvas: {path_csv}')
print(f'   {len(df_out)} linhas | {len(df_out.columns)} colunas')

# ── Erros mais confiantes ─────────────────────────────────────
erros = df_out[df_out['correto'] == 0].nlargest(10, 'confianca')
print('\n⚠️  Top-10 erros mais confiantes:')
print(f'  {"Imagem":<30} {"Real":<18} {"Predito":<18} {"Confiança":>10}')
print('  ' + '-'*78)
for _, row in erros.iterrows():
    print(f'  {str(row["imagem"]):<30} {row["classe_real"]:<18} '
          f'{row["classe_predita"]:<18} {row["confianca"]:>10.4f}')

# ══════════════════════════════════════════════════════════════
#  RESUMO FINAL
# ══════════════════════════════════════════════════════════════

print(f'\n{"="*55}')
print('🏁 INFERÊNCIA CONCLUÍDA — ResNet-50 | DATAP')
print(f'{"="*55}')
print(f'  Imagens avaliadas : {len(all_labels)}')
print(f'  Acurácia          : {acc:.4f} ({acc*100:.1f}%)')
print(f'  F1-Macro          : {f1_macro:.4f}')
print(f'  F1-Weighted       : {f1_weighted:.4f}')
print(f'  AUC-ROC (macro)   : {auc:.4f}')
print(f'  Kappa Quadrático  : {kappa:.4f}')
print(f'{"="*55}')
print(f'\n📁 Resultados em: {OUTPUT_DIR}')

for fname in [
    'inferencia_confusion_matrix.png',
    'inferencia_f1_por_classe.png',
    'inferencia_confianca.png',
    'inferencia_predicoes.csv',
]:
    full = os.path.join(OUTPUT_DIR, fname)
    mark = '✅' if os.path.exists(full) else '❌'
    print(f'  {mark} {fname}')

Mounted at /content/drive
Dispositivo : cuda
GPU         : Tesla T4
VRAM        : 15.6 GB

Config:
  DATAP   : /content/drive/MyDrive/RETINOPATIA/DATAP
  CSV     : /content/drive/MyDrive/RETINOPATIA/DATAP/trainLabels.csv
  Modelo  : /content/drive/MyDrive/RETINOPATIA/resnet50_retinopatia_final.pth
  Saída   : /content/drive/MyDrive/RETINOPATIA/inferencia_datap

PASSO 1 — Lendo trainLabels.csv
Shape   : (35126, 2)
Colunas : ['image', 'level']

Primeiras linhas:
      image  level
0   10_left      0
1  10_right      0
2   13_left      0
3  13_right      0
4   15_left      1

✅ Coluna de imagem detectada : "image"
✅ Coluna de label detectada  : "level"
   Valores únicos de label    : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

PASSO 2 — Mapeando imagens nas pastas DATAP
  ✅ Mild                 label=1     2444 imagens
  ✅ Moderate             label=2     5292 imagens
  ✅ No_DR                label=0    25817 imagens
  ✅ Proliferate_DR       label=4      709 imagens

Inferindo:   0%|          | 0/1098 [00:00<?, ?it/s]